In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]

sys.path.append(str(ROOT))

import pandas as pd

from src.feature_selection.rfe_random_forest import run_rfe_random_forest

In [2]:
data_path = ROOT / "data/processed/ml_dataset.csv"

df = pd.read_csv(data_path)

df["trade_date"] = pd.to_datetime(df["trade_date"])

df.head()

,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


In [3]:
target = "return_5d"

train_df = df[
    (df["trade_date"] >= "2024-03-13") &
    (df["trade_date"] <= "2025-12-31")
].copy()

valid_df = df[
    (df["trade_date"] >= "2026-01-01") &
    (df["trade_date"] <= "2026-06-30")
].copy()

features = [
    col
    for col in df.columns
    if col not in ["stock_code", "trade_date", target]
]

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Feature count:", len(features))
print(features)

Train: (1895, 28)
Validation: (600, 28)
Feature count: 25
['return_1d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']


In [4]:
rfe_rf_results = run_rfe_random_forest(
    train_df=train_df,
    valid_df=valid_df,
    features=features,
    target=target,
)

print(f"총 실험 수: {len(rfe_rf_results)}")

rfe_rf_results.head(10)

[1/405] k=5, step=1, n_estimators=100, max_depth=None, max_features=sqrt
[2/405] k=5, step=1, n_estimators=100, max_depth=None, max_features=log2
[3/405] k=5, step=1, n_estimators=100, max_depth=None, max_features=0.5
[4/405] k=5, step=1, n_estimators=100, max_depth=5, max_features=sqrt
[5/405] k=5, step=1, n_estimators=100, max_depth=5, max_features=log2
[6/405] k=5, step=1, n_estimators=100, max_depth=5, max_features=0.5
[7/405] k=5, step=1, n_estimators=100, max_depth=10, max_features=sqrt
[8/405] k=5, step=1, n_estimators=100, max_depth=10, max_features=log2
[9/405] k=5, step=1, n_estimators=100, max_depth=10, max_features=0.5
[10/405] k=5, step=1, n_estimators=300, max_depth=None, max_features=sqrt
[11/405] k=5, step=1, n_estimators=300, max_depth=None, max_features=log2
[12/405] k=5, step=1, n_estimators=300, max_depth=None, max_features=0.5
[13/405] k=5, step=1, n_estimators=300, max_depth=5, max_features=sqrt
[14/405] k=5, step=1, n_estimators=300, max_depth=5, max_features=log

,method,n_features,step,n_estimators,max_depth,max_features,features,rmse
0,RFE + Random Forest,15,1.0,300,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050424
1,RFE + Random Forest,15,1.0,500,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050466
2,RFE + Random Forest,15,1.0,100,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050483
3,RFE + Random Forest,15,2.0,500,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050606
4,RFE + Random Forest,15,0.1,500,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050606
5,RFE + Random Forest,20,0.1,500,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050616
6,RFE + Random Forest,20,2.0,500,10.0,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050616
7,RFE + Random Forest,15,1.0,500,NaN,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050633
8,RFE + Random Forest,20,2.0,100,NaN,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050666
9,RFE + Random Forest,20,0.1,100,NaN,0.5,"[return_1d, return_10d, return_20d, intraday_r...",0.050666


In [5]:
output_path = ROOT / "data/processed/filter_results/rfe_random_forest_results.csv"

rfe_rf_results.to_csv(
    output_path,
    index=False,
)

print(f"저장 완료: {output_path}")

저장 완료: /Users/yangjaehoon/Desktop/StockLens/data/processed/filter_results/rfe_random_forest_results.csv
